# LoRA Adapter — Character Profile Training

Fine-tunes a TinyLlama LoRA adapter using structured instruction–response pairs
derived from `character_profile_hamlet.json`.

In [ ]:
''' Loading Dependencies '''
import json
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import torch

## Load & Parse the Character Profile JSON

In [ ]:
with open("../data/character_profile_hamlet.json", "r", encoding="utf-8") as f:
    profile = json.load(f)

print(f"Loaded profile for: {profile['character']}")
print(f"Top-level keys: {list(profile.keys())}")

## Build Instruction–Response Training Pairs

Each JSON section is converted into one or more `(instruction, response)` pairs so
the model learns to answer questions *as* or *about* Hamlet in character.

In [ ]:
pairs = []

# Background
pairs.append((
    "Who is Hamlet, and what is his background?",
    profile["background"].replace("\n", " ").strip()
))

# Core traits
for trait in profile["core_traits"]:
    pairs.append((
        f"Describe Hamlet's {trait['name'].lower()} trait.",
        trait["description"]
    ))
    pairs.append((
        f"How does Hamlet exhibit {trait['name'].lower()} behavior?",
        trait["description"]
    ))

# Internal conflicts
for conflict in profile["key_internal_conflicts"]:
    pairs.append((
        f"Explain Hamlet's internal conflict regarding {conflict['name'].lower()}.",
        conflict["description"]
    ))

# Relationships
for rel in profile["relationships"]:
    pairs.append((
        f"How does Hamlet relate to {rel['character']}?",
        rel["description"]
    ))
    pairs.append((
        f"What is {rel['character']}'s role in Hamlet's story?",
        f"{rel['role']}. {rel['description']}"
    ))

# Psychological arc
pairs.append((
    "Describe Hamlet's psychological arc throughout the play.",
    profile["psychological_arc"].replace("\n", " ").strip()
))

# Why Hamlet endures
pairs.append((
    "Why does Hamlet endure as a literary figure?",
    profile["why_hamlet_endures"].replace("\n", " ").strip()
))

# Character analysis summary (split into chunks to avoid extreme length)
summary_text = profile["character_analysis"]["summary"].replace("\n", " ").strip()
pairs.append((
    "Provide a detailed character analysis of Hamlet.",
    summary_text
))

# Bullet-point facts
for bullet in profile["character_analysis"]["bullets"]:
    pairs.append((
        "State one key fact about Hamlet.",
        bullet
    ))

# Roleplay prompts — Hamlet speaks in first person
roleplay_prompts = [
    ("How do you feel about your father's death?",
     "My father's death weighs upon me like a cloak of lead I cannot shed. "
     "The world, once full of promise, has turned into an unweeded garden — "
     "rank and gross in nature. I cannot simply cast off grief as others counsel."),
    ("What do you think of Claudius?",
     "He is the embodiment of corruption — a smiling villain who poisoned "
     "not only my father but the very state of Denmark itself. "
     "To me he is Hyperion to a satyr; my father to this."),
    ("To be, or not to be — what do you mean by that?",
     "I wrestle with the endurance of suffering against the fear of what lies beyond death. "
     "Whether it is nobler to bear the slings and arrows of outrageous fortune, "
     "or to take arms against them — that is the question I cannot cease asking."),
    ("Why do you delay your revenge?",
     "It is not cowardice but conscience. I must be certain — beyond the whisper of a ghost — "
     "that Claudius is guilty. To act rashly is to become the very evil I seek to correct."),
]
pairs.extend(roleplay_prompts)

print(f"Total training pairs: {len(pairs)}")

## Format Using TinyLlama Chat Template

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are Hamlet, Prince of Denmark. You speak with philosophical depth, "
    "melancholic wit, and introspective honesty. Answer as Hamlet would."
)

def format_chat(instruction: str, response: str) -> str:
    """Format a single pair using TinyLlama's chat template tokens."""
    return (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{instruction}</s>\n"
        f"<|assistant|>\n{response}</s>"
    )

formatted_texts = [format_chat(q, a) for q, a in pairs]
print("Example training sample:")
print(formatted_texts[0])

## Load the Model and Configure LoRA

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,                  # Low-rank dimension
    lora_alpha=32,        # Scaling factor
    lora_dropout=0.1,     # Dropout for regularisation
    target_modules=["q_proj", "v_proj"],  # Attention projection layers
)

model.add_adapter(lora_config, adapter_name="hamlet_adapter")
model.set_adapter("hamlet_adapter")
print("LoRA adapter attached.")

## Tokenize the Dataset

In [ ]:
MAX_LENGTH = 512

raw_dataset = Dataset.from_dict({"text": formatted_texts})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_dataset = raw_dataset.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Dataset size: {len(tokenized_dataset)} samples")

## Train the LoRA Adapter

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/lora_hamlet_profile",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 8
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),   # Use fp16 if a GPU is available
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

## Save the Adapter

In [ ]:
model.save_pretrained("../models/lora_hamlet_profile", adapter_name="hamlet_adapter")
tokenizer.save_pretrained("../models/lora_hamlet_profile")
print("Adapter saved to ../models/lora_hamlet_profile")

## Inference — Test the Fine-Tuned Adapter

In [ ]:
# Reload base model + fine-tuned adapter
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
inf_tokenizer = AutoTokenizer.from_pretrained("../models/lora_hamlet_profile")
inf_tokenizer.pad_token = inf_tokenizer.eos_token

inf_model = PeftModel.from_pretrained(
    base_model,
    "../models/lora_hamlet_profile",
    adapter_name="hamlet_adapter"
)
inf_model.eval()

def ask_hamlet(question: str, max_new_tokens: int = 120) -> str:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{question}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = inf_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=inf_tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return inf_tokenizer.decode(new_tokens, skip_special_tokens=True)

test_questions = [
    "How do you feel about your father's death?",
    "Describe your relationship with Ophelia.",
    "What is your greatest internal struggle?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask_hamlet(q)}")
    print("-" * 60)